In [ ]:
import pandas as pd

In [ ]:
# 인코딩 정보를 사용하여 파일 다시 읽기
visitors_2023 = pd.read_csv('서울대공원_입장객_정보_2023년_day_한글복원.csv', encoding='euc-kr')
rainfall_2023 = pd.read_csv('2023과천시_강우량.csv', encoding='euc-kr')

# 데이터 확인
visitors_2023.head(), rainfall_2023.head()


In [ ]:
# 강우량 데이터를 일별로 집계
rainfall_2023['일시'] = pd.to_datetime(rainfall_2023['일시'])
rainfall_2023['date'] = rainfall_2023['일시'].dt.date
daily_rainfall = rainfall_2023.groupby('date')['강수량(mm)'].sum().reset_index()
daily_rainfall['비_여부'] = daily_rainfall['강수량(mm)'].apply(lambda x: '비옴' if x > 0 else '비안옴')

# 입장객 데이터와 강우량 데이터 결합
visitors_2023['date'] = pd.to_datetime(visitors_2023['date']).dt.date
merged_data = pd.merge(visitors_2023, daily_rainfall, on='date', how='left')

# 데이터 분류: 요일별 비 여부로 그룹핑
merged_data['요일'] = merged_data['day']
grouped_data = merged_data.groupby(['요일', '비_여부'])['visitors'].sum().reset_index()

grouped_data

In [ ]:
# 요일별 비 여부로 데이터를 분류(여기를 수정하면 됨)
merged_data['day_group'] = merged_data['요일'].map({
    '월': '월요일~목요일',
    '화': '월요일~목요일',
    '수': '월요일~목요일',
    '목': '월요일~목요일',
    '금': '금요일',
    '토': '토요일',
    '일': '일요일'
})

# 비 여부와 요일 그룹으로 데이터 분류
grouped_data = merged_data.groupby(['비_여부', 'day_group'])

# 결과를 보기 쉽게 요약
split_data = {f"{rain}_{day}": data for (rain, day), data in grouped_data}

# 각 분류된 데이터 표시

split_data

In [ ]:
# 각 분류된 데이터를 print로 표시
for key, df in split_data.items():
    print(f"=== {key} 데이터 ===")
    print(df, "\n")  # 데이터가 많을 수 있으므로 일부만 출력


In [ ]:
data_objects = {}
for key, df in split_data.items():
    data_objects[key] = df

In [ ]:
list(data_objects.keys())

In [ ]:
 data_objects['비안옴_금요일']

In [ ]:
# 2023년 공휴일 및 대체공휴일 정보 설정
holidays_2023 = [
    "2023-01-01", "2023-01-21", "2023-01-22", "2023-01-23",  # 설날 연휴
    "2023-03-01",  # 3.1절
    "2023-05-05",  # 어린이날
    "2023-06-06",  # 현충일
    "2023-08-15",  # 광복절
    "2023-09-28", "2023-09-29", "2023-09-30",  # 추석 연휴
    "2023-10-03",  # 개천절
    "2023-10-09",  # 한글날
    "2023-12-25"   # 크리스마스
]
holidays_2023 = pd.to_datetime(holidays_2023).date

# 공휴일 여부 컬럼 추가
merged_data['공휴일_여부'] = merged_data['date'].apply(lambda x: '공휴일' if x in holidays_2023 else '비공휴일')

# 새로운 기준 추가: 공휴일 데이터 분리
merged_data['category'] = merged_data.apply(
    lambda row: f"{row['비_여부']}_공휴일" if row['공휴일_여부'] == '공휴일' else f"{row['비_여부']}_{row['day_group']}",
    axis=1
)

# 10개 기준으로 데이터 분류
grouped_data = merged_data.groupby('category')

# 데이터 객체로 저장
data_objects = {category: group for category, group in grouped_data}

# 저장된 객체 이름 확인
list(data_objects.keys())

In [ ]:
data_objects

In [ ]:
# 원본 데이터에서 12월 30일 확인
original_dec_30 = visitors_2023[visitors_2023['date'] == pd.to_datetime("2023-12-30").date()]

# 모든 데이터에 대해 요일 재생성
merged_data['day_corrected'] = pd.to_datetime(merged_data['date']).dt.strftime('%a')  # 요일 재생성

# 수정된 데이터에서 12월 30일 및 12월 31일 확인
corrected_dec_30_31 = merged_data[merged_data['date'].isin([
    pd.to_datetime("2023-12-30").date(),
    pd.to_datetime("2023-12-31").date()
])]

original_dec_30, corrected_dec_30_31


In [ ]:
# 12월 30일의 day 값을 '토요일'로 수정
merged_data.loc[merged_data['date'] == pd.to_datetime("2023-12-30").date(), 'day'] = '토'

# 수정된 데이터에 대해 다시 day_group 및 category 설정
merged_data['day_group'] = merged_data['요일'].map({
    '월': '월요일~목요일',
    '화': '월요일~목요일',
    '수': '월요일~목요일',
    '목': '월요일~목요일',
    '금': '금요일',
    '토': '토요일',
    '일': '일요일'
})

merged_data['category'] = merged_data.apply(
    lambda row: f"{row['비_여부']}_공휴일" if row['공휴일_여부'] == '공휴일' else f"{row['비_여부']}_{row['day_group']}",
    axis=1
)

# 수정된 데이터에서 12월 30일 및 12월 31일 확인
corrected_dec_30_31 = merged_data[merged_data['date'].isin([
    pd.to_datetime("2023-12-30").date(),
    pd.to_datetime("2023-12-31").date()
])]

corrected_dec_30_31


In [ ]:
# 12월 30일의 요일 정보(day_corrected)를 기반으로 올바른 day_group 값 설정
merged_data['day_group'] = merged_data['day_corrected'].map({
    'Mon': '월요일~목요일',
    'Tue': '월요일~목요일',
    'Wed': '월요일~목요일',
    'Thu': '월요일~목요일',
    'Fri': '금요일',
    'Sat': '토요일',
    'Sun': '일요일'
})

# category 컬럼도 다시 설정
merged_data['category'] = merged_data.apply(
    lambda row: f"{row['비_여부']}_공휴일" if row['공휴일_여부'] == '공휴일' else f"{row['비_여부']}_{row['day_group']}",
    axis=1
)

# 수정된 데이터에서 12월 30일 및 12월 31일 확인
corrected_dec_30_31 = merged_data[merged_data['date'].isin([
    pd.to_datetime("2023-12-30").date(),
    pd.to_datetime("2023-12-31").date()
])]

corrected_dec_30_31


In [ ]:
# 요일(day_corrected)와 day_group, day 비교를 통해 데이터의 일관성 확인
inconsistencies = merged_data[
    (merged_data['day_corrected'].map({
        'Mon': '월', 'Tue': '화', 'Wed': '수', 'Thu': '목',
        'Fri': '금', 'Sat': '토', 'Sun': '일'
    }) != merged_data['day'])
]

# 데이터 일관성 문제 개수
num_inconsistencies = len(inconsistencies)

# 문제 있었던 데이터 확인
inconsistencies


In [ ]:
# 모든 데이터의 day 값을 day_corrected 기준으로 재설정
merged_data['day'] = merged_data['day_corrected'].map({
    'Mon': '월', 'Tue': '화', 'Wed': '수', 'Thu': '목',
    'Fri': '금', 'Sat': '토', 'Sun': '일'
})

# day_group 및 category 재설정
merged_data['day_group'] = merged_data['day'].map({
    '월': '월요일~목요일',
    '화': '월요일~목요일',
    '수': '월요일~목요일',
    '목': '월요일~목요일',
    '금': '금요일',
    '토': '토요일',
    '일': '일요일'
})

merged_data['category'] = merged_data.apply(
    lambda row: f"{row['비_여부']}_공휴일" if row['공휴일_여부'] == '공휴일' else f"{row['비_여부']}_{row['day_group']}",
    axis=1
)

# 수정 후 데이터의 일관성 확인
inconsistencies_after_fix = merged_data[
    (merged_data['day_corrected'].map({
        'Mon': '월', 'Tue': '화', 'Wed': '수', 'Thu': '목',
        'Fri': '금', 'Sat': '토', 'Sun': '일'
    }) != merged_data['day'])
]

# 문제 해결 여부 확인
len(inconsistencies_after_fix), inconsistencies_after_fix.head()


In [ ]:
merged_data
# day(정확한 요일 데이터) ,요일(이상한 요일 데이터)

In [ ]:
# '요일' 컬럼 삭제
merged_data.drop(columns=['요일'], inplace=True)

# 결과 데이터 확인
merged_data


In [ ]:
# merged_data를 기준으로 10가지 기준으로 데이터 분류
grouped_data = merged_data.groupby('category')

# 10가지 분류 데이터를 개별 객체로 저장
data_objects = {category: group for category, group in grouped_data}

# 저장된 객체 이름 확인
list(data_objects.keys())

In [ ]:
import matplotlib.pyplot as plt

# 꺾은선 그래프 생성
for category, data in data_objects.items():
    plt.figure(figsize=(10, 6))
    plt.plot(data['date'], data['individual_visitors'], marker='o', linestyle='-', label=category)
    plt.title(f"꺾은선 그래프: {category}")
    plt.xlabel("Date")
    plt.ylabel("Visitors")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
######################################################2022년도

In [ ]:
######################################################2022년도

In [ ]:

visitors_2022 = pd.read_csv('서울대공원_입장객_정보_2022년_day_한글복원.csv', encoding='utf-8-sig')
rainfall_2022 = pd.read_csv('2022과천시_강우량.csv', encoding='euc-kr')

# 데이터 확인
visitors_2022.head(), rainfall_2022.head()


In [ ]:
# 강우량 데이터를 일별로 집계
rainfall_2022['일시'] = pd.to_datetime(rainfall_2022['일시'])
rainfall_2022['date'] = rainfall_2022['일시'].dt.date
daily_rainfall_2022 = rainfall_2022.groupby('date')['강수량(mm)'].sum().reset_index()
daily_rainfall_2022['비_여부'] = daily_rainfall_2022['강수량(mm)'].apply(lambda x: '비옴' if x > 0 else '비안옴')

# 입장객 데이터와 강우량 데이터 결합
visitors_2022['date'] = pd.to_datetime(visitors_2022['date']).dt.date
merged_2022 = pd.merge(visitors_2022, daily_rainfall_2022, on='date', how='left')

# 2022년 공휴일 및 대체공휴일 정보 설정
holidays_2022 = [
    "2022-01-01", "2022-01-31", "2022-02-01", "2022-02-02",  # 설날 연휴
    "2022-03-01",  # 3.1절
    "2022-05-05",  # 어린이날
    "2022-06-06",  # 현충일
    "2022-08-15",  # 광복절
    "2022-09-09", "2022-09-10", "2022-09-11",  # 추석 연휴
    "2022-10-03",  # 개천절
    "2022-10-09", "2022-10-10",  # 한글날 및 대체공휴일
    "2022-12-25"   # 크리스마스
]
holidays_2022 = pd.to_datetime(holidays_2022).date

# 공휴일 여부 컬럼 추가
merged_2022['공휴일_여부'] = merged_2022['date'].apply(lambda x: '공휴일' if x in holidays_2022 else '비공휴일')

# 요일 컬럼 추가 및 분류 기준 설정
merged_2022['day_corrected'] = pd.to_datetime(merged_2022['date']).dt.strftime('%a')  # 요일 재생성
merged_2022['day'] = merged_2022['day_corrected'].map({
    'Mon': '월', 'Tue': '화', 'Wed': '수', 'Thu': '목',
    'Fri': '금', 'Sat': '토', 'Sun': '일'
})

merged_2022['day_group'] = merged_2022['day'].map({
    '월': '월요일~목요일',
    '화': '월요일~목요일',
    '수': '월요일~목요일',
    '목': '월요일~목요일',
    '금': '금요일',
    '토': '토요일',
    '일': '일요일'
})

# 최종 분류 기준 설정
merged_2022['category'] = merged_2022.apply(
    lambda row: f"{row['비_여부']}_공휴일" if row['공휴일_여부'] == '공휴일' else f"{row['비_여부']}_{row['day_group']}",
    axis=1
)

# 데이터 분류
grouped_data_2022 = merged_2022.groupby('category')

# 10가지 기준으로 개별 데이터 저장
data_objects_2022 = {category: group for category, group in grouped_data_2022}

# 저장된 객체 이름 확인
list(data_objects_2022.keys())


In [ ]:
data_objects_2022

In [ ]:
import matplotlib.pyplot as plt

# 2022년 꺾은선 그래프 생성
for category, data in data_objects_2022.items():
    plt.figure(figsize=(10, 6))
    plt.plot(data['date'], data['individual_visitors'], marker='o', linestyle='-', label=category)
    plt.title(f"2022년 꺾은선 그래프: {category}")
    plt.xlabel("Date")
    plt.ylabel("Visitors")
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
##################### 2021년도

In [ ]:
########################### 2021년도

In [ ]:

visitors_2021 = pd.read_csv('서울대공원_입장객_정보_2021년_day_한글복원.csv', encoding='utf-8-sig')
rainfall_2021 = pd.read_csv('2021과천시_강우량.csv', encoding='euc-kr')

# 데이터 확인
visitors_2021.head(), rainfall_2021.head()


In [ ]:
# 강우량 데이터를 일별로 집계
rainfall_2021['일시'] = pd.to_datetime(rainfall_2021['일시'])
rainfall_2021['date'] = rainfall_2021['일시'].dt.date
daily_rainfall_2021 = rainfall_2021.groupby('date')['강수량(mm)'].sum().reset_index()
daily_rainfall_2021['비_여부'] = daily_rainfall_2021['강수량(mm)'].apply(lambda x: '비옴' if x > 0 else '비안옴')

# 입장객 데이터와 강우량 데이터 결합
visitors_2021['date'] = pd.to_datetime(visitors_2021['date']).dt.date
merged_2021 = pd.merge(visitors_2021, daily_rainfall_2021, on='date', how='left')

# 2021년 공휴일 및 대체공휴일 정보 설정
holidays_2021 = [
    "2021-01-01", "2021-02-11", "2021-02-12", "2021-02-13",  # 설날 연휴
    "2021-03-01",  # 3.1절
    "2021-05-05",  # 어린이날
    "2021-06-06",  # 현충일
    "2021-08-15", "2021-08-16",  # 광복절 및 대체공휴일
    "2021-09-20", "2021-09-21", "2021-09-22",  # 추석 연휴
    "2021-10-03", "2021-10-04",  # 개천절 및 대체공휴일
    "2021-10-09",  # 한글날
    "2021-12-25"   # 크리스마스
]
holidays_2021 = pd.to_datetime(holidays_2021).date

# 공휴일 여부 컬럼 추가
merged_2021['공휴일_여부'] = merged_2021['date'].apply(lambda x: '공휴일' if x in holidays_2021 else '비공휴일')

# 요일 컬럼 추가 및 분류 기준 설정
merged_2021['day_corrected'] = pd.to_datetime(merged_2021['date']).dt.strftime('%a')  # 요일 재생성
merged_2021['day'] = merged_2021['day_corrected'].map({
    'Mon': '월', 'Tue': '화', 'Wed': '수', 'Thu': '목',
    'Fri': '금', 'Sat': '토', 'Sun': '일'
})

merged_2021['day_group'] = merged_2021['day'].map({
    '월': '월요일~목요일',
    '화': '월요일~목요일',
    '수': '월요일~목요일',
    '목': '월요일~목요일',
    '금': '금요일',
    '토': '토요일',
    '일': '일요일'
})

# 최종 분류 기준 설정
merged_2021['category'] = merged_2021.apply(
    lambda row: f"{row['비_여부']}_공휴일" if row['공휴일_여부'] == '공휴일' else f"{row['비_여부']}_{row['day_group']}",
    axis=1
)

# 데이터 분류
grouped_data_2021 = merged_2021.groupby('category')

# 10가지 기준으로 개별 데이터 저장
data_objects_2021 = {category: group for category, group in grouped_data_2021}

# 저장된 객체 이름 확인
list(data_objects_2021.keys())


In [ ]:
data_objects_2021

In [ ]:
import matplotlib.pyplot as plt

# 2021년 꺾은선 그래프 생성
for category, data in data_objects_2021.items():
    plt.figure(figsize=(10, 6))
    plt.plot(data['date'], data['individual_visitors'], marker='o', linestyle='-', label=category)
    plt.title(f"2021년 꺾은선 그래프: {category}")
    plt.xlabel("Date")
    plt.ylabel("Visitors")
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
################################2020년도

In [ ]:
#########################################2020년도

In [ ]:

visitors_2020 = pd.read_csv('서울대공원_입장객_정보_2020년_day_한글복원_new.csv', encoding='utf-8-sig')

# 데이터 확인
visitors_2020.head()


In [ ]:

rainfall_2020 = pd.read_csv('2020과천시_강우량.csv', encoding='euc-kr')

# 데이터 확인
rainfall_2020.head()


In [ ]:
# 강우량 데이터를 일별로 집계
rainfall_2020['일시'] = pd.to_datetime(rainfall_2020['일시'])
rainfall_2020['date'] = rainfall_2020['일시'].dt.date
daily_rainfall_2020 = rainfall_2020.groupby('date')['강수량(mm)'].sum().reset_index()
daily_rainfall_2020['비_여부'] = daily_rainfall_2020['강수량(mm)'].apply(lambda x: '비옴' if x > 0 else '비안옴')

# 입장객 데이터와 강우량 데이터 결합
visitors_2020['date'] = pd.to_datetime(visitors_2020['date']).dt.date
merged_2020 = pd.merge(visitors_2020, daily_rainfall_2020, on='date', how='left')

# 2020년 공휴일 및 대체공휴일 정보 설정
holidays_2020 = [
    "2020-01-01", "2020-01-24", "2020-01-25", "2020-01-26",  # 설날 연휴
    "2020-03-01",  # 3.1절
    "2020-04-15",  # 선거일
    "2020-05-05",  # 어린이날
    "2020-06-06",  # 현충일
    "2020-08-15",  # 광복절
    "2020-09-30", "2020-10-01", "2020-10-02",  # 추석 연휴
    "2020-10-03",  # 개천절
    "2020-10-09",  # 한글날
    "2020-12-25"   # 크리스마스
]
holidays_2020 = pd.to_datetime(holidays_2020).date

# 공휴일 여부 컬럼 추가
merged_2020['공휴일_여부'] = merged_2020['date'].apply(lambda x: '공휴일' if x in holidays_2020 else '비공휴일')

# 요일 컬럼 추가 및 분류 기준 설정
merged_2020['day_corrected'] = pd.to_datetime(merged_2020['date']).dt.strftime('%a')  # 요일 재생성
merged_2020['day'] = merged_2020['day_corrected'].map({
    'Mon': '월', 'Tue': '화', 'Wed': '수', 'Thu': '목',
    'Fri': '금', 'Sat': '토', 'Sun': '일'
})

merged_2020['day_group'] = merged_2020['day'].map({
    '월': '월요일~목요일',
    '화': '월요일~목요일',
    '수': '월요일~목요일',
    '목': '월요일~목요일',
    '금': '금요일',
    '토': '토요일',
    '일': '일요일'
})

# 최종 분류 기준 설정
merged_2020['category'] = merged_2020.apply(
    lambda row: f"{row['비_여부']}_공휴일" if row['공휴일_여부'] == '공휴일' else f"{row['비_여부']}_{row['day_group']}",
    axis=1
)

# 데이터 분류
grouped_data_2020 = merged_2020.groupby('category')

# 10가지 기준으로 개별 데이터 저장
data_objects_2020 = {category: group for category, group in grouped_data_2020}

# 저장된 객체 이름 확인
list(data_objects_2020.keys())


In [ ]:
data_objects_2020

In [ ]:
import matplotlib.pyplot as plt

# 2020년 꺾은선 그래프 생성
for category, data in data_objects_2020.items():
    plt.figure(figsize=(10, 6))
    plt.plot(data['date'], data['individual_visitors'], marker='o', linestyle='-', label=category)
    plt.title(f"2020년 꺾은선 그래프: {category}")
    plt.xlabel("Date")
    plt.ylabel("Visitors")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
########################2019년도









################################2019년도


In [ ]:
#######################################2019년도

In [ ]:
# 데이터 로드
visitors_2019 = pd.read_csv('서울대공원_입장객_정보_2019년_day_한글복원 (1).csv', encoding='utf-8-sig')
rainfall_2019 = pd.read_csv('2019과천시_강우량.csv', encoding='euc-kr')

# 데이터 확인
visitors_2019.head(), rainfall_2019.head()

In [ ]:
# 강우량 데이터를 일별로 집계
rainfall_2019['일시'] = pd.to_datetime(rainfall_2019['일시'])
rainfall_2019['date'] = rainfall_2019['일시'].dt.date
daily_rainfall_2019 = rainfall_2019.groupby('date')['강수량(mm)'].sum().reset_index()
daily_rainfall_2019['비_여부'] = daily_rainfall_2019['강수량(mm)'].apply(lambda x: '비옴' if x > 0 else '비안옴')

# 입장객 데이터와 강우량 데이터 결합
visitors_2019['date'] = pd.to_datetime(visitors_2019['date']).dt.date
merged_2019 = pd.merge(visitors_2019, daily_rainfall_2019, on='date', how='left')

# 2019년 공휴일 및 대체공휴일 정보 설정
holidays_2019 = [
    "2019-01-01", "2019-02-04", "2019-02-05", "2019-02-06",  # 설날 연휴
    "2019-03-01",  # 3.1절
    "2019-05-05", "2019-05-06",  # 어린이날 및 대체공휴일
    "2019-06-06",  # 현충일
    "2019-08-15",  # 광복절
    "2019-09-12", "2019-09-13", "2019-09-14",  # 추석 연휴
    "2019-10-03",  # 개천절
    "2019-10-09",  # 한글날
    "2019-12-25"   # 크리스마스
]
holidays_2019 = pd.to_datetime(holidays_2019).date

# 공휴일 여부 컬럼 추가
merged_2019['공휴일_여부'] = merged_2019['date'].apply(lambda x: '공휴일' if x in holidays_2019 else '비공휴일')

# 요일 컬럼 추가 및 분류 기준 설정
merged_2019['day_corrected'] = pd.to_datetime(merged_2019['date']).dt.strftime('%a')  # 요일 재생성
merged_2019['day'] = merged_2019['day_corrected'].map({
    'Mon': '월', 'Tue': '화', 'Wed': '수', 'Thu': '목',
    'Fri': '금', 'Sat': '토', 'Sun': '일'
})

merged_2019['day_group'] = merged_2019['day'].map({
    '월': '월요일~목요일',
    '화': '월요일~목요일',
    '수': '월요일~목요일',
    '목': '월요일~목요일',
    '금': '금요일',
    '토': '토요일',
    '일': '일요일'
})

# 최종 분류 기준 설정
merged_2019['category'] = merged_2019.apply(
    lambda row: f"{row['비_여부']}_공휴일" if row['공휴일_여부'] == '공휴일' else f"{row['비_여부']}_{row['day_group']}",
    axis=1
)

# 데이터 분류
grouped_data_2019 = merged_2019.groupby('category')

# 10가지 기준으로 개별 데이터 저장
data_objects_2019 = {category: group for category, group in grouped_data_2019}

# 저장된 객체 이름 확인
list(data_objects_2019.keys())


In [ ]:
data_objects_2019

In [ ]:
import matplotlib.pyplot as plt

# 2019년 꺾은선 그래프 생성
for category, data in data_objects_2019.items():
    plt.figure(figsize=(10, 6))
    plt.plot(data['date'], data['individual_visitors'], marker='o', linestyle='-', label=category)
    plt.title(f"2019년 꺾은선 그래프: {category}")
    plt.xlabel("Date")
    plt.ylabel("Visitors")
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
######################################################## 2018년도!!!

In [ ]:
############################################################## 2018년도!!!

In [ ]:

visitors_2018 = pd.read_csv('서울대공원_입장객_정보_2018년_day_한글복원2.csv', encoding='utf-8-sig')
rainfall_2018 = pd.read_csv('2018과천시_강우량.csv', encoding='euc-kr')

# 데이터 확인
visitors_2018.head(), rainfall_2018.head()

In [ ]:
# 강우량 데이터를 일별로 집계
rainfall_2018['일시'] = pd.to_datetime(rainfall_2018['일시'])
rainfall_2018['date'] = rainfall_2018['일시'].dt.date
daily_rainfall_2018 = rainfall_2018.groupby('date')['강수량(mm)'].sum().reset_index()
daily_rainfall_2018['비_여부'] = daily_rainfall_2018['강수량(mm)'].apply(lambda x: '비옴' if x > 0 else '비안옴')

# 입장객 데이터와 강우량 데이터 결합
visitors_2018['date'] = pd.to_datetime(visitors_2018['date']).dt.date
merged_2018 = pd.merge(visitors_2018, daily_rainfall_2018, on='date', how='left')

# 2018년 공휴일 및 대체공휴일 정보 설정
holidays_2018 = [
    "2018-01-01", "2018-02-15", "2018-02-16", "2018-02-17",  # 설날 연휴
    "2018-03-01",  # 3.1절
    "2018-05-05", "2018-05-07",  # 어린이날 및 대체공휴일
    "2018-06-06",  # 현충일
    "2018-08-15",  # 광복절
    "2018-09-23", "2018-09-24", "2018-09-25",  # 추석 연휴
    "2018-10-03",  # 개천절
    "2018-10-09",  # 한글날
    "2018-12-25"   # 크리스마스
]
holidays_2018 = pd.to_datetime(holidays_2018).date

# 공휴일 여부 컬럼 추가
merged_2018['공휴일_여부'] = merged_2018['date'].apply(lambda x: '공휴일' if x in holidays_2018 else '비공휴일')

# 요일 컬럼 추가 및 분류 기준 설정
merged_2018['day_corrected'] = pd.to_datetime(merged_2018['date']).dt.strftime('%a')  # 요일 재생성
merged_2018['day'] = merged_2018['day_corrected'].map({
    'Mon': '월', 'Tue': '화', 'Wed': '수', 'Thu': '목',
    'Fri': '금', 'Sat': '토', 'Sun': '일'
})

merged_2018['day_group'] = merged_2018['day'].map({
    '월': '월요일~목요일',
    '화': '월요일~목요일',
    '수': '월요일~목요일',
    '목': '월요일~목요일',
    '금': '금요일',
    '토': '토요일',
    '일': '일요일'
})

# 최종 분류 기준 설정
merged_2018['category'] = merged_2018.apply(
    lambda row: f"{row['비_여부']}_공휴일" if row['공휴일_여부'] == '공휴일' else f"{row['비_여부']}_{row['day_group']}",
    axis=1
)

# 데이터 분류
grouped_data_2018 = merged_2018.groupby('category')

# 10가지 기준으로 개별 데이터 저장
data_objects_2018 = {category: group for category, group in grouped_data_2018}

# 저장된 객체 이름 확인
list(data_objects_2018.keys())


In [ ]:
data_objects_2018

In [ ]:
import matplotlib.pyplot as plt

# 2018년 꺾은선 그래프 생성
for category, data in data_objects_2018.items():
    plt.figure(figsize=(10, 6))
    plt.plot(data['date'], data['individual_visitors'], marker='o', linestyle='-', label=category)
    plt.title(f"2018년 꺾은선 그래프: {category}")
    plt.xlabel("Date")
    plt.ylabel("Visitors")
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
import matplotlib.pyplot as plt

# 연도별 데이터 객체를 리스트로 정리
data_objects_all_years = {
    "2023": data_objects,
    "2022": data_objects_2022,
    "2021": data_objects_2021,
    "2020": data_objects_2020,
    "2019": data_objects_2019,
    "2018": data_objects_2018
}

# 비교할 카테고리 리스트
categories = ['비안옴_공휴일',
              '비안옴_금요일',
              '비안옴_월요일~목요일',
              '비안옴_일요일',
              '비안옴_토요일',
              '비옴_공휴일',
              '비옴_금요일',
              '비옴_월요일~목요일',
              '비옴_일요일',
              '비옴_토요일']

# 카테고리별 비교 그래프 생성
for category in categories:
    plt.figure(figsize=(10, 6))
    for year, data_objects in data_objects_all_years.items():
        if category in data_objects:
            data = data_objects[category]
            plt.plot(data['date'], data['individual_visitors'], marker='o', label=f"{year}년")
    plt.title(f"{category} 방문객 비교 (2018-2023)")
    plt.xlabel("Date")
    
    plt.ylabel("Visitors")
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib import rc

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows 환경에서는 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False    # 한글 폰트 사용 시 마이너스 기호 깨짐 방지

# 연도별 데이터 객체를 리스트로 정리
data_objects_all_years = {
    "2023": data_objects,
    "2022": data_objects_2022,
    "2021": data_objects_2021,
    "2020": data_objects_2020,
    "2019": data_objects_2019,
    "2018": data_objects_2018
}

# 비교할 카테고리 리스트
categories = ['비안옴_공휴일',
              '비안옴_금요일',
              '비안옴_월요일~목요일',
              '비안옴_일요일',
              '비안옴_토요일',
              '비옴_공휴일',
              '비옴_금요일',
              '비옴_월요일~목요일',
              '비옴_일요일',
              '비옴_토요일']

# 카테고리별 비교 그래프 생성
for category in categories:
    plt.figure(figsize=(10, 6))
    for year, data_objects in data_objects_all_years.items():
        if category in data_objects:
            data = data_objects[category]
            plt.plot(data['date'], data['individual_visitors'], marker='o', label=f"{year}년")
    plt.title(f"{category} 방문객 비교 (2018-2023)")
    plt.xlabel("날짜")
    plt.ylabel("방문객 수")
    plt.ylim(0,55000)
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
# 1. 왜 공휴일은 저런 방문객 차이가 나왔을까?

# => 2021년도만 유일하게 어린이날때 비가 왔다 공휴일은 날씨의 영향을 받지 않는다 

In [ ]:
# 날씨가 큰영향을 주는게 아니라 월별로 영향을 주는것이다 

In [ ]:
# 이제 날씨는 배제할 생각
# -> 1월~12월 즉 월별로 비교 분석을 해볼 생각이다